In [ ]:
"""
=============================================================================
CLINICAL NUTRITION TARGET PREDICTOR  ·   Dataset Generator
=============================================================================

PURPOSE
-------
Generates a large, clinically realistic synthetic dataset of Type-2 Diabetes
(T2D) patients and their personalised daily macronutrient targets.  The
dataset is intended for training / evaluating ML models that predict:
  • daily_calories_kcal
  • daily_carbohydrates_g
  • daily_protein_g
  • daily_fat_g
  • daily_fiber_g

CLINICAL GUIDELINE SOURCES
---------------------------
  • ADA Standards of Medical Care in Diabetes (2024 / 2025)
    https://diabetesjournals.org/care/issue/47/Supplement_1
    — Nutrition §5: no single optimal macro ratio; carb quality > quantity;
      ≥14 g fiber per 1000 kcal; protein 15–20 % kcal (no CKD);
      low-carb is viable when glycaemia targets are not met.

  • ADA/EASD Nutrition Consensus Report (Evert et al., Diabetes Care 2019)
    https://doi.org/10.2337/dci19-0014
    — Reducing carbohydrate has the most evidence for glycaemia.
    — Very-low-carb: <26 % kcal; Low-carb: 26–44 %; Moderate: 45–65 %.

  • Mifflin–St Jeor BMR equation
    Mifflin MD et al. Am J Clin Nutr. 1990;51(2):241-247.
    https://pubmed.ncbi.nlm.nih.gov/2305711/
    Frankenfield D et al. J Am Diet Assoc. 2005;105(5):775-789.
    — Most accurate predictive equation in overweight/obese adults (73 %
      accuracy within ±10 % vs. indirect calorimetry); recommended by ADA
      and Dietitians of Canada.

  • KDOQI 2020 Nutrition in CKD (NKF / Academy of Nutrition & Dietetics)
    Ikizler TA et al. Am J Kidney Dis. 2020;76(3 Suppl 1):S1-S107.
    https://pubmed.ncbi.nlm.nih.gov/32829751/
    — CKD G3–G5 (eGFR <60): LPD 0.55–0.60 g/kg/d; T2D+CKD ≥0.6–0.8 g/kg/d.
    — CKD G1–G2 (eGFR ≥60): standard 0.8–1.0 g/kg/d.
    — High protein (>1.3 g/kg/d) should be avoided when at risk of progression.

  • KDIGO 2024 CKD Guideline
    Kidney Int. 2024;105(4S):S117-S314.
    https://doi.org/10.1016/j.kint.2023.10.018
    — Recommendation 3.3.1.1: maintain ≥0.8 g/kg/d in CKD G3–G5 (2C).
    — Avoid high protein >1.3 g/kg/d in those at risk of progression.

  • AHA Dietary Fats & CVD Presidential Advisory (Sacks FM et al.)
    Circulation. 2017;136(3):e1-e23.
    https://doi.org/10.1161/CIR.0000000000000510
    — Total fat 20–35 % kcal (AHA); saturated fat <6 % kcal.
    — Replace SFA with PUFA to reduce CVD risk.
    — Very high TG (>500 mg/dL) → restrict fat severely to prevent pancreatitis.
    — High TG (150–499 mg/dL) → moderate fat restriction + ↓ refined carbs.

  • PROT-AGE Consensus (Bauer J et al., JAMDA 2013)
    https://doi.org/10.1016/j.jamda.2013.05.021
    — Adults ≥65 y: protein 1.0–1.2 g/kg/d to prevent sarcopenia;
      higher (1.2–1.5 g/kg/d) with acute/chronic illness.

  • WHO/FAO/UNU Energy and Protein Requirements (1985)
    https://www.fao.org/3/y5686e/y5686e.pdf
    — Energy density: carbohydrate 4 kcal/g, protein 4 kcal/g, fat 9 kcal/g.

  • ADA 2025 §5.24: ≥14 g dietary fiber per 1000 kcal consumed.
    AHA 2006 guidelines: ≥30 g/day dietary fiber for CVD risk.
    Both endorse high-fiber diets for T2D.

  • Roberts & Dallal. Public Health Nutrition. 2005;8(7A):1028-1036.
    Age-related metabolic rate decline (~5 kcal/day per year after age 30).

  • CDC Physical Activity Data (2020):
    https://www.cdc.gov/physicalactivity/data/index.htm
    — Sedentary/light activity ~70 % of T2D clinic population.

  • IDF Diabetes Atlas, 10th edition (2021):
    Slight male predominance in T2D worldwide.

  • NHANES 2017–2020: anthropometrics, glycaemia, lipids, blood pressure.

=============================================================================
"""

# ── Standard library ──────────────────────────────────────────────────────
import os
import json
import warnings
warnings.filterwarnings("ignore")

# ── Numerical / data ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── ML ────────────────────────────────────────────────────────────────────
from sklearn.model_selection  import train_test_split, KFold, cross_val_score
from sklearn.preprocessing    import LabelEncoder, StandardScaler
from sklearn.metrics          import mean_absolute_error, r2_score, mean_squared_error
from sklearn.multioutput      import MultiOutputRegressor
import xgboost as xgb

# ── RAG dependencies ──────────────────────────────────────────────────────
# Install: pip install faiss-cpu sentence-transformers
import faiss
from sentence_transformers import SentenceTransformer

# ── Plotting (optional but recommended) ───────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")   # headless / server-side rendering


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 1 · UTILITY HELPERS
# ═════════════════════════════════════════════════════════════════════════════

def clip(x: float, lo: float, hi: float) -> float:
    """
    Clamp a scalar to the closed interval [lo, hi].
    Using numpy so it handles both Python floats and numpy scalars uniformly.
    """
    return float(np.minimum(np.maximum(x, lo), hi))


def activity_multiplier(level: str) -> float:
    """
    Physical Activity Level (PAL) multipliers for total daily energy expenditure.

    Applied to Mifflin–St Jeor BMR to yield TDEE (Total Daily Energy Expenditure).

    Source: Mifflin MD et al. Am J Clin Nutr. 1990;51(2):241-247.
            WHO/FAO/UNU Expert Consultation 1985 (Table 5.2).

    Sedentary  → desk job / little or no exercise          PAL × 1.20
    Light      → light exercise 1–3 days/week              PAL × 1.375
    Moderate   → moderate exercise 3–5 days/week           PAL × 1.55
    Active     → hard exercise 6–7 days/week               PAL × 1.725

    NOTE: "Very Active" (×1.9, e.g. hard daily training + physical job) is
    intentionally excluded because it is extremely rare in a T2D clinic cohort.
    """
    factors = {
        "Sedentary": 1.20,
        "Light":     1.375,
        "Moderate":  1.55,
        "Active":    1.725,
    }
    if level not in factors:
        raise ValueError(
            f"Unknown activity level '{level}'. "
            f"Valid options: {list(factors.keys())}"
        )
    return factors[level]


def compute_bmr(weight_kg: float, height_cm: float,
                age: int, gender: str) -> float:
    """
    Mifflin–St Jeor Basal Metabolic Rate (kcal / day).

    Validated as the most accurate predictive equation for overweight/obese
    adults — particularly relevant because most T2D patients have elevated BMI.

    Formula (Mifflin MD et al., Am J Clin Nutr 1990):
      Both sexes : 10 × W(kg) + 6.25 × H(cm) − 5 × A(years)
      Male       : + 5
      Female     : − 161

    Accuracy: ~73 % of estimates fall within ±10 % of measured REE by indirect
    calorimetry (Frankenfield D et al., J Am Diet Assoc 2005;105:775-789;
    Van Dessel K et al., Clin Nutr ESPEN 2024;59:422-435).

    Limitation: Slightly under-predicts REE in patients with high visceral
    adiposity or metabolic syndrome.  For dataset generation purposes this
    deterministic formula is appropriate; in clinical practice indirect
    calorimetry is preferred where available.
    """
    base = 10.0 * weight_kg + 6.25 * height_cm - 5.0 * age
    return base + 5.0 if gender == "Male" else base - 161.0


def classify_ckd_stage(egfr: float) -> str:
    """
    KDIGO 2024 CKD G-stage classification by eGFR (mL/min/1.73 m²).

    Source: KDIGO 2024 CKD Guideline, Table 1.
    https://doi.org/10.1016/j.kint.2023.10.018

    G1  : eGFR ≥ 90   — Normal / high (no CKD unless albuminuria present)
    G2  : eGFR 60–89  — Mildly decreased
    G3a : eGFR 45–59  — Mild-to-moderately decreased
    G3b : eGFR 30–44  — Moderately-to-severely decreased
    G4  : eGFR 15–29  — Severely decreased
    G5  : eGFR < 15   — Kidney failure
    """
    if egfr >= 90:
        return "G1"
    elif egfr >= 60:
        return "G2"
    elif egfr >= 45:
        return "G3a"
    elif egfr >= 30:
        return "G3b"
    elif egfr >= 15:
        return "G4"
    else:
        return "G5"


def classify_bmi(bmi: float) -> str:
    """
    WHO BMI classification (adults ≥18 y).
    Source: WHO Technical Report Series 894. 2000.
    https://www.who.int/publications/i/item/obesity-preventing-and-managing-the-global-epidemic

    < 18.5  : Underweight
    18.5–24.9: Normal
    25.0–29.9: Overweight (pre-obese)
    30.0–34.9: Obese class I
    35.0–39.9: Obese class II
    ≥ 40.0  : Obese class III (morbid)
    """
    if bmi < 18.5:
        return "Underweight"
    elif bmi < 25.0:
        return "Normal"
    elif bmi < 30.0:
        return "Overweight"
    elif bmi < 35.0:
        return "Obese_I"
    elif bmi < 40.0:
        return "Obese_II"
    else:
        return "Obese_III"


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 2 · PATIENT FEATURE SAMPLERS
# ═════════════════════════════════════════════════════════════════════════════

def gender_sampler(rng: np.random.Generator) -> str:
    """
    Sex distribution for T2D clinic populations.

    Source: IDF Diabetes Atlas, 10th edition 2021 — slight male predominance.
    NHANES 2017–2020 also shows ~54–56 % male in T2D cohorts.
    """
    return str(rng.choice(["Male", "Female"], p=[0.55, 0.45]))


def choose_activity_level(rng: np.random.Generator) -> str:
    """
    Physical activity distribution in T2D clinic populations.

    Source: CDC National Physical Activity Statistics 2020.
    https://www.cdc.gov/physicalactivity/data/index.htm
    Approximately 70 % of T2D patients are sedentary or only lightly active,
    consistent with large T2D epidemiological cohorts (LOOK AHEAD trial data).

    Probabilities:
      Sedentary : 0.35  (desk job, <1500 steps/day exercise equivalent)
      Light     : 0.35  (light exercise 1–3 d/wk)
      Moderate  : 0.22  (moderate exercise 3–5 d/wk)
      Active    : 0.08  (hard exercise 6–7 d/wk — rare in T2D)
    """
    levels = ["Sedentary", "Light", "Moderate", "Active"]
    probs  = [0.35,         0.35,    0.22,        0.08]
    return str(rng.choice(levels, p=probs))


def primary_goal_sampler(rng: np.random.Generator) -> str:
    """
    Nutrition goal distribution reflecting typical T2D dietitian referral patterns.

    Source: ADA 2024 §5: weight loss and glycaemic control are the two primary
    nutrition goals; maintenance applies to patients already at goal weight.
    LOOK AHEAD trial: ~80 % of T2D patients were overweight/obese at enrolment,
    supporting the high weight-loss referral rate.

    Weight loss    : 0.45
    Maintenance    : 0.35
    Glycemic control: 0.20  (patients at healthy weight but suboptimal HbA1c)
    """
    goals = ["Weight loss", "Maintenance", "Glycemic control"]
    probs = [0.45,           0.35,          0.20]
    return str(rng.choice(goals, p=probs))


def smoking_sampler(rng: np.random.Generator) -> int:
    """
    Smoking prevalence in T2D adults (~22 %).

    Source: WHO Tobacco Fact Sheet 2023 + ADA 2024 §5 Lifestyle Management.
    Smoking is included as a feature because it independently affects
    cardiovascular risk, which influences fat macro targets.
    Returns binary int: 0 = non-smoker, 1 = current smoker.
    """
    return int(rng.choice([0, 1], p=[0.78, 0.22]))


def alcohol_sampler(rng: np.random.Generator) -> int:
    """
    Regular alcohol use prevalence in T2D adults (~20 %).

    Source: ADA 2024 §5 Lifestyle Management.
    ADA: alcohol provides ~7 kcal/g; moderate drinking (≤1 drink/d female,
    ≤2/d male) may affect macro balance but is not contraindicated in T2D
    without other reasons (pancreatitis, high TG, pregnancy).
    Returns binary int: 0 = no regular alcohol, 1 = regular use.
    """
    return int(rng.choice([0, 1], p=[0.80, 0.20]))


def sample_patient(rng: np.random.Generator) -> dict:
    """
    Generates one synthetic but clinically realistic T2D patient record.

    All distributions are calibrated to real-world epidemiological data:
      • NHANES 2017–2020  — anthropometrics, glycaemia, lipids, BP
      • ADA 2024 §6       — glycaemic target ranges
      • NCEP-ATP III / AHA — lipid panel reference ranges
      • KDIGO 2012/2024   — eGFR trajectories in T2D
      • Framingham Heart Study — BP progression with age

    AGE RANGE
    ---------
    T2D occurs predominantly in adults ≥30 y; onset before 18 is classified as
    Type 1 or MODY in most guidelines.  We model ages 18–90 with a distribution
    skewed towards middle-age (peak T2D prevalence 45–75 y per ADA 2024).
    Age range is extended to 18–95 to cover elderly patients who are seen in
    T2D clinics (e.g., LADA diagnosed late, or longstanding T2D in the elderly).

    EDGE CASES EXPLICITLY COVERED
    ------------------------------
    1. Very elderly (≥80 y) — low eGFR, muscle mass loss, protein needs ↑
    2. Young adults (18–35 y) — high eGFR, may be physically active
    3. Normal BMI T2D (~15 % of T2D population; "lean T2D")
    4. Morbidly obese (BMI ≥40) — aggressive caloric deficit required
    5. Advanced CKD (eGFR <30) — strict protein restriction
    6. Very poor glycaemic control (HbA1c ≥10 %) — aggressive carb restriction
    7. Hypertriglyceridaemia (TG >400 mg/dL) — fat restriction critical
    8. Hypertension stage 2 (SBP ≥160) — caloric restriction + DASH
    9. Active smokers with dyslipidaemia — higher CVD risk → tighter fat targets
    10. Goal = Glycemic control in normal-weight patient
    """
    gender = gender_sampler(rng)

    # ── DEMOGRAPHICS ─────────────────────────────────────────────────────────
    # Age: skewed toward middle-age/elderly using a mixture of ranges.
    # T2D prevalence by age (CDC 2022):
    #   18–44 y: ~4 % of T2D;  45–64 y: ~17 %;  ≥65 y: ~29 %.
    # We sample from realistic T2D age distribution.
    age_segment = rng.choice([0, 1, 2, 3], p=[0.05, 0.25, 0.40, 0.30])
    if age_segment == 0:
        age = int(rng.integers(18, 35))    # young adults with T2D (5 %)
    elif age_segment == 1:
        age = int(rng.integers(35, 50))    # middle-age onset (25 %)
    elif age_segment == 2:
        age = int(rng.integers(50, 70))    # peak prevalence (40 %)
    else:
        age = int(rng.integers(70, 96))    # elderly — higher CKD, sarcopenia (30 %)

    # Height: sex-stratified Gaussian, clipped to physiological limits.
    # Source: NHANES 2017–2020 mean heights (adults).
    if gender == "Male":
        height_cm = clip(rng.normal(175, 7), 150, 200)
    else:
        height_cm = clip(rng.normal(162, 6), 140, 185)

    # BMI: T2D population skews toward overweight/obese but includes normal
    # weight (~15 %) and morbidly obese (~8 %).
    # Source: NHANES 2017–2020; ADA 2024 §8 Obesity Management.
    bmi_segment = rng.choice([0, 1, 2, 3, 4],
                              p=[0.05, 0.10, 0.35, 0.32, 0.18])
    if bmi_segment == 0:
        bmi = clip(rng.normal(17.5, 0.8), 14.0, 18.4)  # underweight (rare in T2D)
    elif bmi_segment == 1:
        bmi = clip(rng.normal(22.0, 1.5), 18.5, 24.9)  # normal weight (lean T2D)
    elif bmi_segment == 2:
        bmi = clip(rng.normal(27.5, 1.5), 25.0, 29.9)  # overweight
    elif bmi_segment == 3:
        bmi = clip(rng.normal(32.0, 2.0), 30.0, 34.9)  # obese class I
    else:
        bmi = clip(rng.normal(38.5, 3.0), 35.0, 55.0)  # obese class II/III

    weight_kg = clip(bmi * (height_cm / 100.0) ** 2, 35.0, 200.0)

    # ── LIFESTYLE ─────────────────────────────────────────────────────────────
    activity_level = choose_activity_level(rng)

    # Daily step count correlated with activity level.
    # Source: Tudor-Locke C et al. Int J Behav Nutr Phys Act 2011;8:79.
    # T2D patients: median ~5500 steps/day (NHANES pedometry data).
    step_mu = {
        "Sedentary": 3200,   # <5000 typically defines sedentary
        "Light":     6000,   # 5000–7499
        "Moderate":  9000,   # 7500–9999
        "Active":    12500,  # ≥10000
    }[activity_level]
    steps_per_day = int(clip(rng.normal(step_mu, step_mu * 0.25), 500, 25000))

    # Sleep hours: T2D patients have higher rates of sleep disorders.
    # Source: Cappuccio FP et al. Diabetes Care 2010;33(2):414-420.
    # Mean ~6.5 h/night with higher variance than healthy populations.
    sleep_hours = clip(rng.normal(6.5, 1.2), 3.5, 10.0)

    # ── GLYCAEMIC HISTORY ─────────────────────────────────────────────────────
    # Diabetes duration increases with age (onset typically ~45 y on average).
    # Duration is bounded between 0 (new diagnosis) and 40 years.
    mean_duration = max(0.0, (age - 45.0) / 3.0)
    diabetes_duration = clip(rng.normal(mean_duration, 4.0), 0.0, 40.0)

    # HbA1c: realistic T2D clinic distribution.
    # Source: NHANES 2015–2018 treated T2D cohort — mean ~7.7 %, wide range.
    # ADA targets: <7.0 % (controlled), 7.0–8.0 % (acceptable), ≥8 % (poor).
    hba1c = clip(rng.normal(7.8, 1.3), 5.7, 14.0)

    # Fasting plasma glucose (FPG): derived from HbA1c using the ADA-validated
    # linear relationship.  Rule of thumb: HbA1c 6.5 % ≈ FPG ~126 mg/dL;
    # each 1 % HbA1c increase ≈ +28 mg/dL FPG.
    # Source: Nathan DM et al. Diabetes Care 2008;31(8):1473-1478.
    fasting_glucose = clip(
        rng.normal(70.0 + (hba1c - 5.7) * 28.0, 20.0), 60.0, 400.0
    )

    # 2-hour postprandial glucose (PPG): ADA peak target <180 mg/dL.
    # PPG is approximately 40–60 mg/dL higher than FPG in T2D.
    postprandial_glucose = clip(
        rng.normal(fasting_glucose + 50.0 + (hba1c - 6.0) * 15.0, 30.0),
        70.0, 500.0
    )

    # ── LIPID PANEL ──────────────────────────────────────────────────────────
    # Triglycerides (TG): rise with poor glycaemic control and higher BMI.
    # Normal: <150 mg/dL; Borderline: 150–199; High: 200–499; Very high: ≥500.
    # Source: AHA/ACC 2018 Cholesterol Guideline; NCEP-ATP III.
    triglycerides = clip(
        rng.normal(
            140.0 + (hba1c - 6.5) * 20.0 + (bmi - 25.0) * 4.0, 50.0
        ),
        40.0, 900.0   # ≥500 mg/dL triggers risk of acute pancreatitis
    )

    # LDL cholesterol: rises modestly with BMI and insulin resistance.
    # Source: NHANES 2017–2020 T2D cohort; AHA target <70 mg/dL if CVD.
    ldl = clip(
        rng.normal(110.0 + (bmi - 25.0) * 1.5 + (hba1c - 6.5) * 2.5, 25.0),
        30.0, 300.0
    )

    # HDL cholesterol: inverse relationship with BMI and TG.
    # Source: NCEP-ATP III; low HDL defined as <40 (M) / <50 (F) mg/dL.
    hdl_target = 52.0 if gender == "Female" else 46.0
    hdl = clip(
        rng.normal(hdl_target - (bmi - 25.0) * 0.9 - (triglycerides - 150.0) * 0.03, 10.0),
        20.0, 100.0
    )

    # ── BLOOD PRESSURE ────────────────────────────────────────────────────────
    # Systolic BP: rises with age and BMI (Framingham Heart Study data).
    # ADA 2024 §10 targets: SBP <130 mmHg for most T2D patients.
    # JNC 8 / AHA/ACC 2017: hypertension ≥130/80 mmHg.
    systolic = clip(
        rng.normal(115.0 + max(age - 40, 0) * 0.65 + (bmi - 25.0) * 0.8, 15.0),
        80.0, 220.0
    )
    diastolic = clip(
        rng.normal(74.0 + max(age - 40, 0) * 0.25 + (bmi - 25.0) * 0.45, 10.0),
        50.0, 130.0
    )

    # ── RENAL FUNCTION ────────────────────────────────────────────────────────
    # eGFR declines with age and diabetes duration.
    # Source: KDIGO 2024; CKD-EPI equation underpins values.
    # Average decline: ~1 mL/min/1.73m²/year in general adults;
    # ~3–5 mL/min/1.73m²/year in T2D with proteinuria (UKPDS data).
    # Younger patients (<40 y) often have hyperfiltration (eGFR >120) early in T2D.
    egfr_base = 120.0 - max(age - 25, 0) * 0.75 - diabetes_duration * 1.2
    egfr = clip(rng.normal(egfr_base, 18.0), 5.0, 140.0)

    # ── OTHER RISK FACTORS ────────────────────────────────────────────────────
    smoking = smoking_sampler(rng)
    alcohol = alcohol_sampler(rng)
    goal    = primary_goal_sampler(rng)

    return {
        "age":                          int(age),
        "gender":                       gender,
        "height_cm":                    round(float(height_cm), 1),
        "weight_kg":                    round(float(weight_kg), 1),
        "bmi":                          round(float(bmi), 1),
        "physical_activity_level":      activity_level,
        "steps_per_day":                int(steps_per_day),
        "sleep_hours":                  round(float(sleep_hours), 1),
        "diabetes_duration_years":      round(float(diabetes_duration), 1),
        "hba1c_percent":                round(float(hba1c), 2),
        "fasting_glucose_mg_dl":        round(float(fasting_glucose), 1),
        "postprandial_glucose_mg_dl":   round(float(postprandial_glucose), 1),
        "triglycerides_mg_dl":          round(float(triglycerides), 1),
        "ldl_cholesterol_mg_dl":        round(float(ldl), 1),
        "hdl_cholesterol_mg_dl":        round(float(hdl), 1),
        "systolic_bp_mmhg":             round(float(systolic), 1),
        "diastolic_bp_mmhg":            round(float(diastolic), 1),
        "egfr_ml_min_1_73m2":           round(float(egfr), 1),
        "smoking_status":               int(smoking),
        "alcohol_use":                  int(alcohol),
        "primary_goal":                 goal,
    }


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 3 · CLINICAL RULE ENGINE — TARGET COMPUTATION
# ═════════════════════════════════════════════════════════════════════════════

def compute_targets(patient: dict) -> dict:
    """
    Deterministic, evidence-based clinical rule engine that converts patient
    features into personalised daily macronutrient targets.

    The logic follows the clinical decision pathway a Registered Dietitian
    Nutritionist (RDN) would use for Medical Nutrition Therapy (MNT) in T2D:

      Step 1 → Estimate TDEE (Mifflin–St Jeor × PAL)
      Step 2 → Apply physiological adjustments (age, steps, BP, BMI)
      Step 3 → Apply goal-based caloric adjustment
      Step 4 → Determine carbohydrate ratio (glycaemia-driven)
      Step 5 → Determine protein ratio (renal-stage-driven, age-adjusted)
      Step 6 → Determine fat ratio (residual, lipid-panel-adjusted)
      Step 7 → Convert ratios to grams; apply absolute safety clamps
      Step 8 → Compute fiber target (ADA 14 g/1000 kcal minimum + TG/HDL add)

    All thresholds and multipliers are cited to published clinical guidelines
    documented in the module docstring.

    Returns
    -------
    dict with five keys (all float, rounded to 1 decimal place):
      daily_calories_kcal, daily_carbohydrates_g, daily_protein_g,
      daily_fat_g, daily_fiber_g
    """
    # ── UNPACK ─────────────────────────────────────────────────────────────────
    age            = patient["age"]
    gender         = patient["gender"]
    weight_kg      = patient["weight_kg"]
    height_cm      = patient["height_cm"]
    bmi            = patient["bmi"]

    activity_level = patient["physical_activity_level"]
    steps          = patient["steps_per_day"]
    sleep_hours    = patient["sleep_hours"]

    hba1c               = patient["hba1c_percent"]
    fasting_glucose     = patient["fasting_glucose_mg_dl"]
    postprandial_glucose= patient["postprandial_glucose_mg_dl"]
    diabetes_duration   = patient["diabetes_duration_years"]

    tg  = patient["triglycerides_mg_dl"]
    ldl = patient["ldl_cholesterol_mg_dl"]
    hdl = patient["hdl_cholesterol_mg_dl"]

    sys_bp = patient["systolic_bp_mmhg"]
    dia_bp = patient["diastolic_bp_mmhg"]
    egfr   = patient["egfr_ml_min_1_73m2"]

    alcohol = patient["alcohol_use"]
    smoking = patient["smoking_status"]
    goal    = patient["primary_goal"]

    ckd_stage  = classify_ckd_stage(egfr)
    bmi_class  = classify_bmi(bmi)

    # ═══════════════════════════════════════════════════════════════════════
    # A)  C A L O R I E S
    # ═══════════════════════════════════════════════════════════════════════

    # Step A1: Mifflin–St Jeor BMR × Physical Activity Level (PAL)
    # This gives the Total Daily Energy Expenditure (TDEE) before adjustments.
    bmr      = compute_bmr(weight_kg, height_cm, age, gender)
    calories = bmr * activity_multiplier(activity_level)

    # Step A2: Age-related metabolic decline (~5 kcal/day per year after 30).
    # Source: Roberts SB & Dallal GE. Public Health Nutr 2005;8(7A):1028-1036.
    # This accounts for declining lean muscle mass and mitochondrial efficiency.
    age_penalty = max(age - 30, 0) * 5.0
    calories -= age_penalty

    # Step A3: Step-count activity bonus.
    # Patients walking >10,000 steps/day burn meaningfully more calories.
    # Smooth ramp: 10k → 20k steps adds 0–12 % extra TDEE.
    # Source: Tudor-Locke C et al. IJBNPA 2011;8:79 — step/energy dose-response.
    if steps > 10000:
        bonus_pct = clip((steps - 10000) / 10000, 0.0, 1.0)
        calories *= (1.0 + 0.12 * bonus_pct)

    # Step A4: Sleep deprivation adjustment.
    # Short sleep (<6 h) is associated with increased appetite and ~5–8 %
    # higher ad-libitum intake; conversely, oversleeping has minimal metabolic
    # effect.  We apply a modest 3 % reduction for very short sleepers as a
    # conservative diet prescription safeguard.
    # Source: Spiegel K et al. Ann Intern Med 2004;141(11):846-850.
    #         St-Onge MP et al. Obesity 2012;20(2):258-264.
    if sleep_hours < 5.5:
        calories *= 0.97   # short sleep → prescribe slightly lower calories

    # Step A5: Hypertension energy restriction.
    # AHA/ACC 2017 and DASH trial: even a 500 kcal/day deficit in overweight
    # hypertensive patients lowers SBP by 3–5 mmHg.
    # Apply 5 % reduction when Stage 2 hypertension is present.
    hypertensive = (sys_bp >= 140) or (dia_bp >= 90)
    if hypertensive:
        calories *= 0.95

    # Step A6: BMI-class-based obesity correction (ADA 2024 §8).
    # ADA endorses 500–750 kcal/day deficit for weight loss in T2D.
    # We implement as proportional multipliers calibrated to BMI class.
    if bmi_class == "Underweight":
        calories *= 1.10   # underweight T2D → slight surplus to prevent wasting
    elif bmi_class == "Normal":
        pass               # no BMI-based adjustment needed
    elif bmi_class == "Overweight":
        calories *= 0.97   # modest deficit
    elif bmi_class == "Obese_I":
        calories *= 0.93   # class I obesity → moderate deficit (~400–500 kcal)
    elif bmi_class == "Obese_II":
        calories *= 0.87   # class II obesity → larger deficit (~600 kcal)
    elif bmi_class == "Obese_III":
        calories *= 0.82   # class III (morbid) → aggressive deficit; close
                            # supervision required per ADA 2024

    # Step A7: Goal-based adjustment.
    # ADA 2024 §5: 400–500 kcal deficit for weight loss; slight restriction
    # (≈3 %) for glycaemic control even without weight-loss goal.
    if goal == "Weight loss":
        calories -= 450.0  # ~0.4–0.5 kg/week deficit (ADA 2024 §8)
    elif goal == "Glycemic control":
        calories *= 0.97   # mild restriction improves postprandial glucose

    # Step A8: Hard physiological clamp.
    # VLED (Very Low Energy Diet) threshold: <800 kcal/day requires
    # medical supervision and is not prescribed here.
    # Minimum 1000 kcal/day for general MNT; maximum 4000 kcal/day.
    # ADA 2024: does not endorse <1200 kcal/day diets without specialist review.
    calories = clip(calories, 1000.0, 4000.0)

    # ═══════════════════════════════════════════════════════════════════════
    # B)  C A R B O H Y D R A T E S
    # ═══════════════════════════════════════════════════════════════════════
    #
    # ADA Nutrition Consensus Report (Evert AB et al. Diabetes Care 2019):
    #   "Reducing overall carbohydrate intake has demonstrated the most
    #    evidence for improving glycemia."
    #
    # Carb categories by % kcal (Evert 2019):
    #   Very-low-carb : <26 %   (ketogenic; viable for select T2D; ADA 2024)
    #   Low-carb      : 26–44 %
    #   Moderate-carb : 45–65 % (standard DRI)
    #
    # Our baseline is 48 % — the mid-point of the low-to-moderate range,
    # appropriate for a typical T2D patient with HbA1c ~7.5–8.0 %.
    # We then apply evidence-based downward adjustments for:
    #   (i)  rising HbA1c     → β-cell function ↓ → less carb tolerance
    #   (ii) glucose excursions → acute glycaemic burden
    #   (iii) long diabetes duration → blunted insulin secretion

    carb_ratio = 0.48 - (hba1c - 6.5) * 0.045

    # HbA1c threshold penalties:
    # ADA 2024 / Evert 2019: low-carb (<44 %) is "viable" for HbA1c not at
    # target; very-low-carb (<26 %) may be used for HbA1c ≥9 % in select pts.
    if hba1c >= 10.0:
        carb_ratio *= 0.80   # very poor control → push toward very-low-carb
    elif hba1c >= 9.0:
        carb_ratio *= 0.87   # poor control → low-carb range (<44 %)
    elif hba1c >= 8.0:
        carb_ratio *= 0.93   # moderately poor control

    # Acute glucose excursion penalties (independent of HbA1c):
    # High fasting glucose indicates insufficient overnight insulin action.
    # High PPG indicates inadequate mealtime carb-to-insulin ratio.
    if fasting_glucose > 180.0:
        carb_ratio *= 0.88
    elif fasting_glucose > 140.0:
        carb_ratio *= 0.93

    if postprandial_glucose > 250.0:
        carb_ratio *= 0.88
    elif postprandial_glucose > 200.0:
        carb_ratio *= 0.93

    # Long diabetes duration: β-cell reserve is progressively lower.
    # After 15 years, most patients have significant β-cell exhaustion
    # (UKPDS progressive loss study, Turner RC et al. JAMA 1999).
    if diabetes_duration > 20.0:
        carb_ratio *= 0.88
    elif diabetes_duration > 10.0:
        carb_ratio *= 0.93

    # Goal modifier:
    if goal == "Glycemic control":
        carb_ratio *= 0.92  # direct glycaemic goal → push further into low-carb

    # Safety clamp: 20–55 % of calories from carbohydrate.
    # Lower bound 20 %: below this ketogenesis is likely without medical
    # supervision; ADA 2024 §5 cautions against ketogenic diets in patients
    # on SGLT-2 inhibitors due to risk of euglycaemic DKA.
    # Upper bound 55 %: consistent with ADA moderate-carb definition.
    carb_ratio = clip(carb_ratio, 0.20, 0.55)

    # ═══════════════════════════════════════════════════════════════════════
    # C)  P R O T E I N
    # ═══════════════════════════════════════════════════════════════════════
    #
    # KDOQI 2020 / KDIGO 2024 protein recommendations by eGFR stage:
    #
    #  eGFR ≥90  (G1)  : 0.8–1.0 g/kg/d  → ratio 0.18–0.20
    #  eGFR 60–89 (G2) : 0.8–1.0 g/kg/d  → ratio 0.18 (conservative)
    #  eGFR 45–59 (G3a): 0.55–0.80 g/kg/d→ ratio 0.15–0.16
    #  eGFR 30–44 (G3b): 0.55–0.60 g/kg/d→ ratio 0.13–0.14
    #  eGFR 15–29 (G4) : 0.55–0.60 g/kg/d→ ratio 0.12–0.13
    #  eGFR  <15  (G5) : 0.55–0.60 g/kg/d→ ratio 0.12 (maximal restriction)
    #
    # KDIGO 2024 Practice Point 3.3.1.1:
    #   Avoid high protein (>1.3 g/kg/d) in adults at risk of CKD progression.
    #
    # ADA 2024 §11 Microvascular Complications:
    #   In diabetic kidney disease without dialysis, target 0.8 g/kg/d.
    #
    # PROT-AGE Consensus (Bauer J et al., JAMDA 2013):
    #   Adults ≥65 y: target 1.0–1.2 g/kg/d; diseased elderly: 1.2–1.5 g/kg/d.
    #   This is HIGHER than younger adults due to anabolic resistance.

    if ckd_stage == "G5":
        protein_ratio = 0.12   # KDOQI: 0.55–0.60 g/kg; maximal restriction
    elif ckd_stage == "G4":
        protein_ratio = 0.13   # KDOQI: 0.55–0.60 g/kg; close supervision
    elif ckd_stage == "G3b":
        protein_ratio = 0.14   # KDOQI: 0.55–0.60 g/kg; moderate restriction
    elif ckd_stage == "G3a":
        protein_ratio = 0.16   # KDOQI: 0.55–0.80 g/kg; cautious
    elif ckd_stage == "G2":
        protein_ratio = 0.18   # KDIGO: ≥0.8 g/kg; mild CKD — standard intake
    else:
        protein_ratio = 0.20   # G1 / normal — standard 0.8–1.0 g/kg/d DRI

    # Older adults require more dietary protein to counteract anabolic resistance.
    # PROT-AGE: ≥65 y → +1–2 % protein ratio; diseased ≥75 y → further +1 %.
    # We must NOT override CKD restrictions; only apply bonus when eGFR is
    # adequate to tolerate higher protein load (G1/G2 = eGFR ≥60).
    if age >= 75 and ckd_stage in ("G1", "G2"):
        protein_ratio += 0.02   # diseased very elderly → 1.2–1.5 g/kg/d
    elif age >= 65 and ckd_stage in ("G1", "G2"):
        protein_ratio += 0.01   # elderly → 1.0–1.2 g/kg/d

    # Weight-loss goal without CKD: slightly higher protein preserves lean mass.
    # Source: Layman DK et al. J Nutr 2003;133(2):411-417;
    #         ADA 2024 §5.23: "attention to preventing protein insufficiency."
    if goal == "Weight loss" and ckd_stage in ("G1", "G2"):
        protein_ratio += 0.01

    # Safety clamp: 12–25 % of calories from protein.
    # Lower bound 12 %: below this in CKD risks protein-energy wasting (PEW).
    # Upper bound 25 %: above 1.5–1.6 g/kg/d provides no additional anabolic
    # benefit and increases renal solute load (ADA 2024).
    protein_ratio = clip(protein_ratio, 0.12, 0.25)

    # ═══════════════════════════════════════════════════════════════════════
    # D)  F A T
    # ═══════════════════════════════════════════════════════════════════════
    #
    # Fat fills the remaining calories after carb + protein allocation.
    # AHA guideline range: 20–35 % of total kcal from fat (total fat).
    # AHA 2017 Presidential Advisory: saturated fat <6 % kcal;
    # Replace with PUFA/MUFA for maximum CVD risk reduction.
    #
    # Triglyceride-specific adjustments (AHA / NCEP-ATP III):
    #   TG 150–199  (borderline-high): moderate fat restriction
    #   TG 200–499  (high):            significant fat restriction + ↓ refined carbs
    #   TG ≥500     (very high):       STRICT fat restriction (<15 % kcal)
    #                                  to prevent chylomicronemia/pancreatitis
    #   Source: Miller M et al. Circulation 2011;123(20):2292-2333 (AHA TG statement)
    #
    # LDL-specific adjustments (AHA 2018 Cholesterol Guideline):
    #   LDL 130–159 (borderline-high): moderate fat reduction
    #   LDL ≥160    (high):            significant fat reduction
    #   LDL ≥190    (very high):       maximal saturated fat restriction

    fat_ratio = 1.0 - carb_ratio - protein_ratio

    # Triglyceride-based fat adjustments:
    if tg >= 500.0:
        # Very severe hypertriglyceridaemia: risk of acute pancreatitis.
        # Fat MUST be restricted to <20 % kcal; some guidelines say <15 %.
        # Source: Schaefer EJ et al. Am J Cardiol 2002;90(8 Suppl):17i-24i.
        fat_ratio = min(fat_ratio, 0.18)
    elif tg >= 300.0:
        fat_ratio *= 0.78   # significant TG elevation → substantial fat cut
    elif tg >= 200.0:
        fat_ratio *= 0.86   # high TG → moderate fat restriction
    elif tg >= 150.0:
        fat_ratio *= 0.94   # borderline-high TG → slight restriction

    # LDL-based fat adjustments:
    if ldl >= 190.0:
        fat_ratio *= 0.80   # very high LDL: maximal saturated fat restriction
    elif ldl >= 160.0:
        fat_ratio *= 0.87   # high LDL
    elif ldl >= 130.0:
        fat_ratio *= 0.94   # borderline-high LDL

    # Low HDL adjustment: low HDL is a CVD risk factor in T2D.
    # Paradoxically, very low-fat diets can worsen HDL; moderate fat (MUFA/PUFA)
    # with high fiber is more beneficial.
    # Source: AHA Dietary Guidelines; Sacks FM et al. Circulation 2017.
    hdl_low_threshold = 50.0 if gender == "Female" else 40.0
    if hdl < hdl_low_threshold:
        # Do NOT further restrict fat; instead ensure ratio stays at least 0.25
        # so unsaturated fat intake is adequate to raise HDL.
        fat_ratio = max(fat_ratio, 0.25)

    # Alcohol adjustment: alcohol contributes 7 kcal/g, displacing fat calories.
    # ADA 2024 §5: moderate alcohol OK; we reduce fat slightly to maintain
    # total energy balance without exceeding caloric target.
    if alcohol == 1:
        fat_ratio *= 0.96

    # Smoking + dyslipidaemia: compounded CVD risk requires tighter fat control.
    # Source: AHA 2017 Presidential Advisory.
    if smoking == 1 and (ldl > 130.0 or tg > 150.0):
        fat_ratio *= 0.95

    # Ensure fat ratio does not exceed the residual after carbs+protein are set.
    # This prevents macro ratios summing > 1.
    max_fat = max(0.0, 1.0 - carb_ratio - protein_ratio)
    fat_ratio = min(fat_ratio, max_fat)

    # AHA range clamp: 20–35 % total fat (35 % upper to allow MUFA-rich diets).
    # Exception: allow <20 % only in severe hypertriglyceridaemia (TG ≥500).
    fat_lower = 0.15 if tg >= 500.0 else 0.20
    fat_ratio = clip(fat_ratio, fat_lower, 0.38)

    # ═══════════════════════════════════════════════════════════════════════
    # E)  CONVERT RATIOS → GRAMS
    # ═══════════════════════════════════════════════════════════════════════
    # Atwater energy factors (FAO/WHO/UNU 1985):
    #   Carbohydrate : 4 kcal / g
    #   Protein      : 4 kcal / g
    #   Fat          : 9 kcal / g

    carbs_g   = calories * carb_ratio    / 4.0
    protein_g = calories * protein_ratio / 4.0
    fat_g     = calories * fat_ratio     / 9.0

    # ── RENAL ABSOLUTE PROTEIN SAFEGUARD ────────────────────────────────
    # Regardless of ratio, clamp protein to absolute g/kg limits per KDOQI 2020
    # and KDIGO 2024 Practice Point 3.3.1.1.
    # These override the ratio-based value to ensure clinical safety.
    #
    #  eGFR <30  (G3b/G4/G5): 0.55–0.60 g/kg/d (LPD; KDOQI 2020 Grade 1A)
    #  eGFR 30–59 (G3):       0.60–0.80 g/kg/d (T2D+CKD; ADA 2024 §11)
    #  eGFR 60–89 (G2):       0.80–1.00 g/kg/d
    #  eGFR ≥90  (G1):        0.80–1.20 g/kg/d
    #  Elderly (≥65, no adv CKD): 1.00–1.20 g/kg/d (PROT-AGE)
    if ckd_stage in ("G4", "G5"):
        prot_min_kg, prot_max_kg = 0.55, 0.60
    elif ckd_stage == "G3b":
        prot_min_kg, prot_max_kg = 0.55, 0.70
    elif ckd_stage == "G3a":
        prot_min_kg, prot_max_kg = 0.60, 0.80
    elif ckd_stage == "G2":
        prot_min_kg, prot_max_kg = 0.80, 1.00
    else:  # G1
        if age >= 65:
            prot_min_kg, prot_max_kg = 1.00, 1.20  # PROT-AGE elderly
        else:
            prot_min_kg, prot_max_kg = 0.80, 1.20

    protein_g = clip(protein_g, prot_min_kg * weight_kg, prot_max_kg * weight_kg)

    # ── UNDERWEIGHT SAFEGUARD ────────────────────────────────────────────
    # Underweight patients (BMI <18.5) should not receive a caloric deficit.
    # Ensure carbs are not excessively restricted for underweight patients.
    if bmi_class == "Underweight":
        # Ensure at minimum 40 % kcal from carbs even if HbA1c is high,
        # to prevent hypoglycaemia and support weight restoration.
        carbs_min_g = calories * 0.40 / 4.0
        carbs_g = max(carbs_g, carbs_min_g)

    # ═══════════════════════════════════════════════════════════════════════
    # F)  F I B E R
    # ═══════════════════════════════════════════════════════════════════════
    #
    # ADA 2025 §5.24 recommendation:
    #   "Emphasize minimally processed, nutrient-dense, high-fiber sources of
    #    carbohydrate (at least 14 g fiber per 1,000 kcal)."
    #
    # AHA 2006 guidelines: ≥30 g/day fiber for CVD risk reduction.
    #
    # Additional evidence-based increments:
    #   + HbA1c: soluble fiber (β-glucan, psyllium) slows glucose absorption.
    #     Source: Weickert MO & Pfeiffer AFH. J Nutr 2008;138(3):439-442.
    #   + High TG: soluble fiber lowers TG by modulating hepatic VLDL secretion.
    #     Source: Brown L et al. Am J Clin Nutr 1999;69(1):30-42 (meta-analysis).
    #   + Low HDL: fiber modestly raises HDL via bile acid sequestration.
    #     Source: Glore SR et al. J Am Diet Assoc 1994;94(4):425-436.
    #   + Activity: active patients have higher GI transit time and typically
    #     tolerate and benefit more from high fiber intake.

    # Base fiber: 14 g per 1000 kcal (ADA 2025 minimum), minimum 25 g/day.
    fiber_base = max(calories * 14.0 / 1000.0, 25.0)

    # Glycaemia increment: +2 g per 1 % HbA1c above 7.0 %
    fiber_hba1c_bonus = max(hba1c - 7.0, 0.0) * 2.0

    # TG increment: +5 g for high TG (β-glucan specifically lowers TG)
    fiber_tg_bonus = 5.0 if tg > 200.0 else (3.0 if tg > 150.0 else 0.0)

    # HDL increment: +3 g for low HDL
    fiber_hdl_bonus = 3.0 if hdl < hdl_low_threshold else 0.0

    # Activity increment: more physically active patients benefit from
    # additional fiber to support microbiome diversity and satiety.
    fiber_activity_bonus = {
        "Sedentary": 0.0,
        "Light":     1.0,
        "Moderate":  2.0,
        "Active":    3.0,
    }[activity_level]

    fiber_g = (fiber_base + fiber_hba1c_bonus + fiber_tg_bonus
               + fiber_hdl_bonus + fiber_activity_bonus)

    # Practical upper limit 60 g/day: above this GI symptoms become common;
    # lower bound is ADA minimum = 14 g/1000 kcal ≈ 20–28 g for most patients.
    fiber_g = clip(fiber_g, max(calories * 14.0 / 1000.0, 20.0), 60.0)

    return {
        "daily_calories_kcal":      round(calories, 1),
        "daily_carbohydrates_g":    round(carbs_g, 1),
        "daily_protein_g":          round(protein_g, 1),
        "daily_fat_g":              round(fat_g, 1),
        "daily_fiber_g":            round(fiber_g, 1),
    }


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 4 · DATASET GENERATOR
# ═════════════════════════════════════════════════════════════════════════════

def generate_dataset(
    n_rows: int  = 10000,
    seed:   int  = 42,
    out_csv: str = "clinical_synthetic_nutrition.csv",
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Generates a large synthetic clinical nutrition dataset.

    The generator is designed to ensure:
      • Realistic distributional coverage across all patient sub-groups
      • No degenerate cases (e.g. protein > calories; fat < 0)
      • All edge cases handled by the clinical rule engine
      • Macro ratios always sum to ≤1.0 (remaining calories are treated as
        alcohol, thermic effect of food, or implicit dietary fat)

    Parameters
    ----------
    n_rows   : number of synthetic patient records (default 10 000)
    seed     : numpy random seed for reproducibility
    out_csv  : output CSV filepath
    verbose  : print progress and summary statistics

    Returns
    -------
    pd.DataFrame with shape (n_rows, 21 features + 5 targets + 2 derived cols)
    Derived columns: ckd_stage_label, bmi_class_label (for EDA / stratification)
    """
    rng  = np.random.default_rng(seed)
    rows = []

    if verbose:
        print(f"Generating {n_rows:,} synthetic T2D patient records …")

    for i in range(n_rows):
        patient = sample_patient(rng)
        targets = compute_targets(patient)

        # Add derived/diagnostic columns for EDA and stratification
        derived = {
            "ckd_stage_label": classify_ckd_stage(patient["egfr_ml_min_1_73m2"]),
            "bmi_class_label": classify_bmi(patient["bmi"]),
        }
        rows.append({**patient, **targets, **derived})

        if verbose and (i + 1) % 2000 == 0:
            print(f"  {i + 1:,} / {n_rows:,} completed")

    df = pd.DataFrame(rows)

    # ── ENFORCE COLUMN ORDER ────────────────────────────────────────────────
    feature_cols = [
        "age", "gender", "height_cm", "weight_kg", "bmi",
        "physical_activity_level", "steps_per_day", "sleep_hours",
        "diabetes_duration_years", "hba1c_percent",
        "fasting_glucose_mg_dl", "postprandial_glucose_mg_dl",
        "triglycerides_mg_dl", "ldl_cholesterol_mg_dl", "hdl_cholesterol_mg_dl",
        "systolic_bp_mmhg", "diastolic_bp_mmhg",
        "egfr_ml_min_1_73m2",
        "smoking_status", "alcohol_use", "primary_goal",
    ]
    target_cols = [
        "daily_calories_kcal",
        "daily_carbohydrates_g",
        "daily_protein_g",
        "daily_fat_g",
        "daily_fiber_g",
    ]
    derived_cols = ["ckd_stage_label", "bmi_class_label"]

    df = df[feature_cols + target_cols + derived_cols]
    df.to_csv(out_csv, index=False)

    if verbose:
        print(f"\n✅  Dataset saved → {out_csv}")
        print(f"    Shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")
        print()
        print("    ── Feature distributions ──────────────────────────────────")
        print(f"    Age          : {df['age'].min()}–{df['age'].max()} y  "
              f"(mean {df['age'].mean():.1f})")
        print(f"    BMI          : {df['bmi'].min():.1f}–{df['bmi'].max():.1f}  "
              f"(mean {df['bmi'].mean():.1f})")
        print(f"    HbA1c        : {df['hba1c_percent'].min():.1f}–"
              f"{df['hba1c_percent'].max():.1f} %  "
              f"(mean {df['hba1c_percent'].mean():.2f})")
        print(f"    eGFR         : {df['egfr_ml_min_1_73m2'].min():.1f}–"
              f"{df['egfr_ml_min_1_73m2'].max():.1f}  "
              f"(mean {df['egfr_ml_min_1_73m2'].mean():.1f})")
        print(f"    TG           : {df['triglycerides_mg_dl'].min():.0f}–"
              f"{df['triglycerides_mg_dl'].max():.0f}  "
              f"(mean {df['triglycerides_mg_dl'].mean():.0f})")
        print()
        print("    ── CKD stage distribution ────────────────────────────────")
        ckd_counts = df["ckd_stage_label"].value_counts().sort_index()
        for stage, cnt in ckd_counts.items():
            print(f"      {stage:<4}  : {cnt:>6,}  ({cnt/len(df)*100:.1f} %)")
        print()
        print("    ── BMI class distribution ────────────────────────────────")
        bmi_counts = df["bmi_class_label"].value_counts()
        for cls, cnt in bmi_counts.items():
            print(f"      {cls:<12}: {cnt:>6,}  ({cnt/len(df)*100:.1f} %)")
        print()
        print("    ── Target variable summary (mean ± std) ──────────────────")
        for col in target_cols:
            print(f"      {col:<30}: "
                  f"{df[col].mean():.1f} ± {df[col].std():.1f}  "
                  f"[{df[col].min():.1f} – {df[col].max():.1f}]")

    return df


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 5 · QUICK SANITY CHECKS
# ═════════════════════════════════════════════════════════════════════════════

def run_sanity_checks(df: pd.DataFrame) -> None:
    """
    Validates that the generated dataset satisfies key clinical constraints.
    Raises AssertionError if any constraint is violated.
    """
    print("\nRunning clinical sanity checks …")

    # 1. Calories within physiological bounds
    assert df["daily_calories_kcal"].between(999, 4001).all(), \
        "Calories out of range [1000, 4000]"

    # 2. Protein g/kg within KDOQI/KDIGO bounds
    prot_per_kg = df["daily_protein_g"] / df["weight_kg"]
    assert (prot_per_kg >= 0.50).all(), \
        f"Some patients have protein < 0.50 g/kg: min={prot_per_kg.min():.2f}"
    assert (prot_per_kg <= 1.50).all(), \
        f"Some patients have protein > 1.50 g/kg: max={prot_per_kg.max():.2f}"

    # 3. Macro ratios sum ≤ 1.0 (with a tiny tolerance for float arithmetic)
    macro_sum_ratio = (
        df["daily_carbohydrates_g"] * 4 +
        df["daily_protein_g"] * 4 +
        df["daily_fat_g"] * 9
    ) / df["daily_calories_kcal"]
    assert (macro_sum_ratio <= 1.05).all(), \
        f"Macros exceed calories: max ratio={macro_sum_ratio.max():.3f}"

    # 4. Fiber at minimum ADA standard (≥14 g / 1000 kcal)
    fiber_density = df["daily_fiber_g"] / (df["daily_calories_kcal"] / 1000.0)
    assert (fiber_density >= 13.5).all(), \
        f"Fiber below ADA minimum: min density={fiber_density.min():.1f} g/1000 kcal"

    # 5. Fat ≤ 35 % calories (AHA upper bound) — except when TG ≥ 500
    fat_pct = df["daily_fat_g"] * 9 / df["daily_calories_kcal"]
    high_tg_mask = df["triglycerides_mg_dl"] < 500.0
    assert (fat_pct[high_tg_mask] <= 0.40).all(), \
        f"Fat above 40 % in normal-TG patients: max={fat_pct[high_tg_mask].max():.3f}"

    # 6. Carbs ≥ 20 % calories (ADA lower bound; prevents unsafe ketosis risk)
    carb_pct = df["daily_carbohydrates_g"] * 4 / df["daily_calories_kcal"]
    assert (carb_pct >= 0.19).all(), \
        f"Carbs below 19 % in some patients: min={carb_pct.min():.3f}"

    print("  ✅  All clinical sanity checks passed.")
    print(f"  Protein range: {prot_per_kg.min():.2f}–{prot_per_kg.max():.2f} g/kg/d")
    print(f"  Fat range:     {fat_pct.min():.1%}–{fat_pct.max():.1%} of kcal")
    print(f"  Carb range:    {carb_pct.min():.1%}–{carb_pct.max():.1%} of kcal")
    print(f"  Fiber density: {fiber_density.min():.1f}–{fiber_density.max():.1f} g/1000 kcal")


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 6 · MAIN
# ═════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    df = generate_dataset(
        n_rows  = 10_000,
        seed    = 42,
        out_csv = "userData.csv",
        verbose = True,
    )
    run_sanity_checks(df)

Generating 10,000 synthetic T2D patient records …
  2,000 / 10,000 completed
  4,000 / 10,000 completed
  6,000 / 10,000 completed
  8,000 / 10,000 completed
  10,000 / 10,000 completed

✅  Dataset saved → userData.csv
    Shape  : 10,000 rows × 28 columns

    ── Feature distributions ──────────────────────────────────
    Age          : 18–95 y  (mean 60.3)
    BMI          : 15.4–48.0  (mean 30.0)
    HbA1c        : 5.7–14.0 %  (mean 7.82)
    eGFR         : 5.0–140.0  (mean 85.8)
    TG           : 40–412  (mean 186)

    ── CKD stage distribution ────────────────────────────────
      G1    :  4,598  (46.0 %)
      G2    :  3,766  (37.7 %)
      G3a   :    982  (9.8 %)
      G3b   :    470  (4.7 %)
      G4    :    151  (1.5 %)
      G5    :     33  (0.3 %)

    ── BMI class distribution ────────────────────────────────
      Overweight  :  3,506  (35.1 %)
      Obese_I     :  3,182  (31.8 %)
      Obese_II    :  1,277  (12.8 %)
      Normal      :    976  (9.8 %)
      Obese_III 

AssertionError: Macros exceed calories: max ratio=1.158